# Exercise — Convert Enriched Iranian Web Logs to Parquet

Convert the GeoIP-enriched Iranian web logs to Parquet and compare the storage size with CSV.

The data dependency is: Exercise 071 creates `weblog-ip-mapping.csv`, and Exercise 072 joins that mapping to the parsed logs to create the input used here.

**Input**: `C:/data/ir-weblogs/weblog-enriched.csv`  
**Output**: `C:/data/ir-weblogs/weblog-enriched.parquet`

## Learning objectives

- load an enriched CSV dataset with pandas;
- inspect and improve data types before serialization;
- write and read a compressed Parquet file with PyArrow;
- validate the CSV-to-Parquet round trip;
- compare CSV and Parquet file sizes.

## 1. Setup

Pandas needs a Parquet engine. Install the dependencies if required:

```powershell
python -m pip install pandas pyarrow
```

In [ ]:
from pathlib import Path

import pandas as pd
import pyarrow as pa

DATA_DIR = Path(r"C:\data\ir-weblogs")
CSV_PATH = DATA_DIR / "weblog-enriched.csv"
PARQUET_PATH = DATA_DIR / "weblog-enriched.parquet"

if not CSV_PATH.is_file():
    raise FileNotFoundError(
        f"Required enriched log file not found: {CSV_PATH}. "
        "Run Exercises 071 and 072 first."
    )

print(f"pandas version: {pd.__version__}")
print(f"PyArrow version: {pa.__version__}")
print(f"Input: {CSV_PATH}")
print(f"Output: {PARQUET_PATH}")

## 2. Load and inspect the enriched logs

Load the complete CSV. Verify that the location columns added from the Exercise 071 mapping are present.

In [ ]:
enriched_logs = pd.read_csv(CSV_PATH, low_memory=False)

required_geo_columns = ["country", "state", "city"]
missing_geo_columns = [
    column for column in required_geo_columns if column not in enriched_logs.columns
]
if missing_geo_columns:
    raise ValueError(
        f"Input is not the expected enriched dataset; missing: {missing_geo_columns}"
    )

print(f"Rows: {len(enriched_logs):,}")
print(f"Columns: {len(enriched_logs.columns)}")
display(enriched_logs.head())
display(enriched_logs.dtypes.rename("dtype").to_frame())

## 3. Prepare efficient data types

Repeated text values such as geographic names are good candidates for pandas' `category` type. Parquet can dictionary-encode these columns efficiently. Missing values remain missing.

In [ ]:
for column in required_geo_columns:
    enriched_logs[column] = enriched_logs[column].astype("category")

print(enriched_logs[required_geo_columns].dtypes)

## 4. Write compressed Parquet

Use Snappy compression for a practical balance of read/write speed and storage reduction. Do not store the pandas index because it is not part of the web-log data.

In [ ]:
PARQUET_PATH.parent.mkdir(parents=True, exist_ok=True)
enriched_logs.to_parquet(
    PARQUET_PATH,
    engine="pyarrow",
    compression="snappy",
    index=False,
)

print(f"Created {PARQUET_PATH}")

## 5. Validate the Parquet round trip

Reload the generated file and verify its shape, column order, and values.

In [ ]:
parquet_logs = pd.read_parquet(PARQUET_PATH, engine="pyarrow")

assert PARQUET_PATH.is_file()
assert parquet_logs.shape == enriched_logs.shape
assert parquet_logs.columns.tolist() == enriched_logs.columns.tolist()
pd.testing.assert_frame_equal(parquet_logs, enriched_logs)

print(f"Validated {len(parquet_logs):,} rows and {len(parquet_logs.columns)} columns.")

## 6. Compare CSV and Parquet sizes

The reduction percentage is positive when Parquet is smaller. Results depend on the data and compression settings.

In [ ]:
csv_bytes = CSV_PATH.stat().st_size
parquet_bytes = PARQUET_PATH.stat().st_size
reduction_pct = (1 - parquet_bytes / csv_bytes) * 100 if csv_bytes else 0.0

size_comparison = pd.DataFrame({
    "format": ["CSV", "Parquet (Snappy)"],
    "bytes": [csv_bytes, parquet_bytes],
    "megabytes": [csv_bytes / 1024**2, parquet_bytes / 1024**2],
})
size_comparison["megabytes"] = size_comparison["megabytes"].round(2)

display(size_comparison)
print(f"Parquet size as a percentage of CSV: {parquet_bytes / csv_bytes * 100:.1f}%")
print(f"Storage reduction: {reduction_pct:.1f}%")

## 7. Optional: demonstrate column projection

Parquet readers can load only requested columns, which is useful for analytical workloads.

In [ ]:
geo_sample = pd.read_parquet(
    PARQUET_PATH,
    engine="pyarrow",
    columns=required_geo_columns,
).head(10)
display(geo_sample)

## Submission checklist

- `weblog-enriched.csv` from Exercises 071–072 is loaded.
- The `country`, `state`, and `city` columns are verified.
- `weblog-enriched.parquet` is written with Snappy compression and no index.
- The Parquet round trip is validated.
- CSV and Parquet sizes are shown in bytes and MiB.
- The percentage storage reduction is calculated.